# MLOps Lab 3 — Model Management & Model Registry with MLflow

**Dataset:** `kidney_disease_cleaned.csv`

### Learning Objectives
- Save and load models.
- Understand MLflow Model vs Model Registry.
- Register a model and create versions.
- Compare registered versions.
- Add tags/metadata.
- Use a Model Registry alias.
- Load the selected registered model and predict.

## 1. Why Model Registry?

A file such as:

```text
kidney_model.pkl
```

is only a saved model.

In MLOps we may have:

```text
KidneyDiseaseClassifier
 ├── Version 1
 ├── Version 2
 └── Version 3
```

The **Model Registry** provides a central place to manage registered models, versions, metadata and aliases.

```text
Experiment
   ↓
Best Run
   ↓
MLflow Model
   ↓
Model Registry
   ↓
Version → Select → Alias → Load/Deploy
```

## 2. Local Registry Setup

For this lab we use SQLite.

Install:

```bash
python -m pip install mlflow scikit-learn pandas
```

The notebook uses:

```text
sqlite:///mlflow_registry.db
```

For the browser UI, use a separate terminal:

```bash
mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow_registry.db --default-artifact-root ./mlartifacts
```

Open:

`http://127.0.0.1:5000`

> Use a database-backed tracking store for this registry practical.

In [ ]:
import pandas as pd
import mlflow
import mlflow.sklearn

from mlflow import MlflowClient
from mlflow.models import infer_signature

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

print("Libraries imported.")

In [ ]:
# Configure SQLite as the local tracking and registry backend.
mlflow.set_tracking_uri("sqlite:///mlflow_registry.db")
mlflow.set_registry_uri("sqlite:///mlflow_registry.db")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Registry URI:", mlflow.get_registry_uri())

In [ ]:
# Load and prepare the dataset.
df = pd.read_csv("kidney_disease_cleaned.csv")

X = df.drop("classification", axis=1).copy()
y = df["classification"].astype(str).str.strip()

if "id" in X.columns:
    X = X.drop("id", axis=1)

num_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

mlflow.set_experiment("Kidney Disease - Model Registry")

## 3. Train Model Version 1

We start with Logistic Regression using `C=1.0`.

In [ ]:
model_v1 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(C=1.0, max_iter=1000))
])

model_v1.fit(X_train, y_train)
pred1 = model_v1.predict(X_test)

metrics_v1 = {
    "accuracy": accuracy_score(y_test, pred1),
    "precision": precision_score(y_test, pred1, average="weighted"),
    "recall": recall_score(y_test, pred1, average="weighted"),
    "f1_score": f1_score(y_test, pred1, average="weighted")
}

print(metrics_v1)

## 4. Log + Register Version 1

`registered_model_name` tells MLflow to create the registered model if it does not exist and create a version under that name.

The input example helps MLflow infer a model signature.

In [ ]:
REGISTERED_NAME = "KidneyDiseaseClassifier"

with mlflow.start_run(run_name="Kidney Disease - Version 1") as run:

    mlflow.log_params({
        "model": "Logistic Regression",
        "C": 1.0,
        "random_state": 42
    })
    mlflow.log_metrics(metrics_v1)

    signature = infer_signature(
        X_train,
        model_v1.predict(X_train)
    )

    model_info_v1 = mlflow.sklearn.log_model(
        model_v1,
        name="kidney_disease_model",
        signature=signature,
        input_example=X_train.head(3),
        serialization_format="cloudpickle",
        registered_model_name=REGISTERED_NAME
    )

    run_id_v1 = run.info.run_id

print("Run:", run_id_v1)
print("Model URI:", model_info_v1.model_uri)

## 5. Inspect the Registry

A registered model can have multiple versions.

In [ ]:
client = MlflowClient()

versions = client.search_model_versions(
    f"name='{REGISTERED_NAME}'"
)

for v in sorted(versions, key=lambda x: int(x.version)):
    print(
        f"Version={v.version} | "
        f"Run={v.run_id} | "
        f"Status={v.status}"
    )

## 6. Create Version 2

Change the Logistic Regression hyperparameter:

```text
C = 0.1
```

Register it using the **same registered model name**.

In [ ]:
model_v2 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(C=0.1, max_iter=1000))
])

model_v2.fit(X_train, y_train)
pred2 = model_v2.predict(X_test)

metrics_v2 = {
    "accuracy": accuracy_score(y_test, pred2),
    "precision": precision_score(y_test, pred2, average="weighted"),
    "recall": recall_score(y_test, pred2, average="weighted"),
    "f1_score": f1_score(y_test, pred2, average="weighted")
}

print(metrics_v2)

In [ ]:
with mlflow.start_run(run_name="Kidney Disease - Version 2") as run:

    mlflow.log_params({
        "model": "Logistic Regression",
        "C": 0.1,
        "random_state": 42
    })
    mlflow.log_metrics(metrics_v2)

    signature = infer_signature(
        X_train,
        model_v2.predict(X_train)
    )

    model_info_v2 = mlflow.sklearn.log_model(
        model_v2,
        name="kidney_disease_model",
        signature=signature,
        input_example=X_train.head(3),
        serialization_format="cloudpickle",
        registered_model_name=REGISTERED_NAME
    )

    run_id_v2 = run.info.run_id

print("Run:", run_id_v2)
print("Model URI:", model_info_v2.model_uri)

## 7. Compare Registered Versions

In [ ]:
version_results = pd.DataFrame([
    {"version": 1, "C": 1.0, **metrics_v1},
    {"version": 2, "C": 0.1, **metrics_v2}
])

display(version_results)

best = version_results.loc[
    version_results["f1_score"].idxmax()
]

best_version = str(int(best["version"]))

print("Selected version:", best_version)

## 8. Add Registry Metadata

Metadata helps people understand what a model version represents.

In [ ]:
client.update_registered_model(
    name=REGISTERED_NAME,
    description="Kidney disease classification model for MLOps classroom demonstration."
)

client.set_model_version_tag(
    name=REGISTERED_NAME,
    version="1",
    key="model_type",
    value="logistic_regression"
)

client.set_model_version_tag(
    name=REGISTERED_NAME,
    version="2",
    key="model_type",
    value="logistic_regression"
)

print("Metadata added.")

## 9. Model Alias — `champion`

An alias gives a meaningful name to the version currently selected for use.

Instead of applications using:

```text
models:/KidneyDiseaseClassifier/2
```

they can use:

```text
models:/KidneyDiseaseClassifier@champion
```

If the selected version changes, move the alias.

In [ ]:
client.set_registered_model_alias(
    REGISTERED_NAME,
    "champion",
    best_version
)

print(f"'champion' -> version {best_version}")

## 10. Load the Registered Model

Now we load the model from the registry instead of a `.pkl` file.

In [ ]:
champion_uri = f"models:/{REGISTERED_NAME}@champion"

loaded_model = mlflow.sklearn.load_model(champion_uri)

print("Loaded:", champion_uri)

predictions = loaded_model.predict(X_test)

print("Accuracy:",
      accuracy_score(y_test, predictions))

print("First 10 predictions:")
print(predictions[:10])

## 11. Saving vs Registry

### Save

```python
import joblib
joblib.dump(model, "model.pkl")
```

Creates a file.

### Load

```python
model = joblib.load("model.pkl")
```

Loads a file.

### Registry

```text
Registered Model
 ├── Version 1
 ├── Version 2
 └── Version 3
```

The registry manages model versions and metadata and provides a stable way to refer to a selected version, such as an alias.

## 12. Explore the MLflow UI

Open:

`http://127.0.0.1:5000`

Inspect:

### Runs
- Parameters
- Metrics
- Model artifacts

### Models / Model Registry
- Registered model
- Versions
- Tags
- Alias `champion`

### Important distinction

**Model Logging** = store the MLflow model with a run.

**Model Registry** = manage registered model versions and lifecycle metadata.

## 13. Student Practice

### Task 1 — Version 3
Train a Random Forest and register it under:

```text
KidneyDiseaseClassifier
```

### Task 2 — Compare
Create a table containing:

- Version
- Model
- Hyperparameters
- Accuracy
- Precision
- Recall
- F1

### Task 3 — Alias
Move `champion` to the best version.

### Task 4 — Prediction
Load:

```text
models:/KidneyDiseaseClassifier@champion
```

and predict on new/test data.

### Questions
1. What is a model version?
2. Why use the same registered model name?
3. What is the difference between saving and registering?
4. What is a model alias?
5. Why is metadata useful?
6. Which version would you deploy and why?

# End of Lab

```text
Train
  ↓
Evaluate
  ↓
Log Model
  ↓
Register
  ↓
Version 1 / Version 2 / ...
  ↓
Compare
  ↓
Select Best
  ↓
Alias: champion
  ↓
Load
  ↓
Prediction / Deployment
```

### Unit-2 connection

```text
Data Versioning (DVC)
        ↓
Feature Engineering / Feast
        ↓
Experiment Tracking / MLflow
        ↓
Hyperparameter Tuning
        ↓
Model Management / Registry
        ↓
Deployment
```